In [ ]:
# MediaCloud Data Collection

from mediacloud.api import SearchApi, DirectoryApi
from datetime import date
import pandas as pd
import time

API_TOKEN = 'XXXXXX'

search_api = SearchApi(API_TOKEN)

# Source IDs
SOURCES = {
    'CNN': 1095,
    'Fox News': 1092,
    'Newsmax': 25349,
    'MSNBC': 1177971,
    'AP': 106145,
    'BBC': 932549,
}

# Expanded keywords per topic
TOPICS = {
    'AI': ['artificial intelligence', 'AI', 'ChatGPT', 'OpenAI'],
    'Iran': ['Iran'],
    'Climate': ['climate change', 'global warming', 'carbon emissions', 'climate crisis'],
    'Immigration': ['immigration', 'border', 'migrants', 'asylum', 'deportation'],
    'Economy': ['economy', 'inflation', 'recession', 'jobs report', 'unemployment'],
}

# Date range
START_DATE = date(2026, 1, 1)
END_DATE = date(2026, 4, 22)

total_queries = sum(len(kw) for kw in TOPICS.values()) * len(SOURCES)
print(f"Sources: {len(SOURCES)}")
print(f"Topics: {len(TOPICS)} ({sum(len(kw) for kw in TOPICS.values())} keywords)")
print(f"Queries: {total_queries}")
print(f"Date range: {START_DATE} to {END_DATE}")

Sources: 6
Topics: 5 (19 keywords)
Queries: 114
Date range: 2026-01-01 to 2026-04-22


In [2]:
# Test token is working

print("Testing API connection...")
try:
    profile = search_api.user_profile()
    print(f"Token valid! Profile: {profile}")
except Exception as e:
    print(f"Token error: {e}")
    print("\nGet a fresh token from: https://search.mediacloud.org/account")

Testing API connection...
Token valid! Profile: {'id': 35519, 'username': 'jwood19', 'is_staff': False, 'is_superuser': False, 'groups': ['api_access'], 'quota': {'provider': 'onlinenews-mediacloud', 'hits': 21, 'week': '2026-04-20', 'limit': 4000}}


In [3]:
# Collect articles with rate limiting

all_stories = []
seen_urls = set()  # Deduplicate across keywords
DELAY = 15  # increased to 15 seconds between requests

for topic_name, keywords in TOPICS.items():
    print(f"\n=== {topic_name.upper()} ===")
    for source_name, source_id in SOURCES.items():
        source_stories = []
        for keyword in keywords:
            try:
                stories, token = search_api.story_list(
                    query=keyword,
                    start_date=START_DATE,
                    end_date=END_DATE,
                    source_ids=[source_id]
                )
                
                for s in stories:
                    url = s.url if hasattr(s, 'url') else s.get('url')
                    if url and url not in seen_urls:
                        seen_urls.add(url)
                        source_stories.append({
                            'topic': topic_name,
                            'keyword': keyword,
                            'source_name': source_name,
                            'source_id': source_id,
                            'title': s.title if hasattr(s, 'title') else s.get('title'),
                            'url': url,
                            'publish_date': s.publish_date if hasattr(s, 'publish_date') else s.get('publish_date'),
                            'language': s.language if hasattr(s, 'language') else s.get('language'),
                        })
                
                print(f"    {source_name:12} [{keyword:25}] +{len(stories)} stories")
                        
            except Exception as e:
                print(f"    {source_name:12} [{keyword:25}] Error: rate limited, waiting 60s...")
                time.sleep(60)  # Extra wait on error
            
            time.sleep(DELAY)
        
        all_stories.extend(source_stories)
        print(f"  {source_name:12} TOTAL: {len(source_stories)} unique stories")

print(f"\n\nTotal unique stories collected: {len(all_stories)}")


=== AI ===
    CNN          [artificial intelligence  ] +166 stories
    CNN          [AI                       ] +962 stories
    CNN          [ChatGPT                  ] +80 stories
    CNN          [OpenAI                   ] Error: rate limited, waiting 60s...
  CNN          TOTAL: 1025 unique stories
    Fox News     [artificial intelligence  ] +217 stories
    Fox News     [AI                       ] +514 stories
    Fox News     [ChatGPT                  ] +48 stories
    Fox News     [OpenAI                   ] Error: rate limited, waiting 60s...
  Fox News     TOTAL: 572 unique stories
    Newsmax      [artificial intelligence  ] +138 stories
    Newsmax      [AI                       ] +163 stories
    Newsmax      [ChatGPT                  ] Error: rate limited, waiting 60s...
    Newsmax      [OpenAI                   ] +32 stories
  Newsmax      TOTAL: 199 unique stories
    MSNBC        [artificial intelligence  ] +0 stories
    MSNBC        [AI                       ] +

In [4]:
# Save metadata

df = pd.DataFrame(all_stories)
df.to_csv('mediacloud_articles.csv', index=False)

print(f"Saved {len(df)} articles to mediacloud_articles.csv")
print("\nBy source:")
print(df.groupby('source_name').size().sort_values(ascending=False))
print("\nBy topic:")
print(df.groupby('topic').size().sort_values(ascending=False))

Saved 16497 articles to mediacloud_articles.csv

By source:
source_name
BBC         5254
AP          3855
Fox News    3627
CNN         2969
Newsmax      792
dtype: int64

By topic:
topic
Immigration    5290
AI             4084
Iran           3702
Economy        2919
Climate         502
dtype: int64


In [ ]:
# Retry failed keywords and append to existing data

# Failed combinations from the log (keyword, source_name, source_id, topic)
FAILED = [
    ('OpenAI', 'CNN', 1095, 'AI'),
    ('OpenAI', 'Fox News', 1092, 'AI'),
    ('ChatGPT', 'Newsmax', 25349, 'AI'),
    ('ChatGPT', 'MSNBC', 1177971, 'AI'), 
    ('ChatGPT', 'AP', 106145, 'AI'),
    ('ChatGPT', 'BBC', 932549, 'AI'),
    ('Iran', 'Newsmax', 25349, 'Iran'),
    ('climate change', 'CNN', 1095, 'Climate'),
    ('climate change', 'Fox News', 1092, 'Climate'),
    ('climate crisis', 'Fox News', 1092, 'Climate'),
    ('climate change', 'MSNBC', 1177971, 'Climate'),
    ('climate change', 'AP', 106145, 'Climate'),
    ('climate change', 'BBC', 932549, 'Climate'),
    ('immigration', 'CNN', 1095, 'Immigration'),
    ('deportation', 'CNN', 1095, 'Immigration'),
    ('migrants', 'Fox News', 1092, 'Immigration'),
    ('immigration', 'MSNBC', 1177971, 'Immigration'),
    ('deportation', 'MSNBC', 1177971, 'Immigration'),
    ('border', 'Newsmax', 25349, 'Immigration'),
    ('asylum', 'AP', 106145, 'Immigration'),
    ('border', 'BBC', 932549, 'Immigration'),
    ('economy', 'CNN', 1095, 'Economy'),
    ('unemployment', 'CNN', 1095, 'Economy'),
    ('jobs report', 'Fox News', 1092, 'Economy'),
    ('inflation', 'Newsmax', 25349, 'Economy'),
    ('economy', 'MSNBC', 1177971, 'Economy'),
    ('unemployment', 'MSNBC', 1177971, 'Economy'),
    ('jobs report', 'AP', 106145, 'Economy'),
    ('recession', 'BBC', 932549, 'Economy'),
]

print(f"Retrying {len(FAILED)} failed queries...")
print("Will retry on rate limit errors until success\n")

retry_stories = []
seen_urls = set()

# Load existing URLs to avoid duplicates
existing = pd.read_csv('mediacloud_articles.csv')
seen_urls = set(existing['url'].dropna().tolist())
print(f"Loaded {len(seen_urls)} existing URLs to avoid duplicates\n")

for keyword, source_name, source_id, topic in FAILED:
    success = False
    attempt = 0
    
    while not success:
        attempt += 1
        try:
            stories, token = search_api.story_list(
                query=keyword,
                start_date=START_DATE,
                end_date=END_DATE,
                source_ids=[source_id]
            )
            
            new_count = 0
            for s in stories:
                url = s.url if hasattr(s, 'url') else s.get('url')
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    retry_stories.append({
                        'topic': topic,
                        'keyword': keyword,
                        'source_name': source_name,
                        'source_id': source_id,
                        'title': s.title if hasattr(s, 'title') else s.get('title'),
                        'url': url,
                        'publish_date': s.publish_date if hasattr(s, 'publish_date') else s.get('publish_date'),
                        'language': s.language if hasattr(s, 'language') else s.get('language'),
                    })
                    new_count += 1
            
            print(f"  ✓ {source_name:12} [{keyword:20}] +{new_count} new stories")
            success = True
            time.sleep(15)  # Normal delay after success
                    
        except Exception as e:
            wait_time = 60 * attempt  # Increase wait time with each attempt
            print(f"  ⏳ {source_name:12} [{keyword:20}] Rate limited, waiting {wait_time}s (attempt {attempt})...")
            time.sleep(wait_time)

print(f"\n\nRetried queries: {len(retry_stories)} new stories")

# Append to existing CSV
if retry_stories:
    retry_df = pd.DataFrame(retry_stories)
    combined = pd.concat([existing, retry_df], ignore_index=True)
    combined.to_csv('mediacloud_articles.csv', index=False)
    print(f"Appended to mediacloud_articles.csv - now {len(combined)} total stories")

Retrying 29 failed queries...
Will retry on rate limit errors until success

Loaded 16497 existing URLs to avoid duplicates

  ✓ CNN          [OpenAI              ] +7 new stories
  ✓ Fox News     [OpenAI              ] +3 new stories
  ✓ Newsmax      [ChatGPT             ] +0 new stories
  ⏳ MSNBC        [ChatGPT             ] Rate limited, waiting 60s (attempt 1)...
  ✓ MSNBC        [ChatGPT             ] +0 new stories
  ✓ AP           [ChatGPT             ] +4 new stories
  ✓ BBC          [ChatGPT             ] +51 new stories
  ⏳ Newsmax      [Iran                ] Rate limited, waiting 60s (attempt 1)...
  ✓ Newsmax      [Iran                ] +639 new stories
  ✓ CNN          [climate change      ] +66 new stories
  ✓ Fox News     [climate change      ] +49 new stories
  ✓ Fox News     [climate crisis      ] +11 new stories
  ⏳ MSNBC        [climate change      ] Rate limited, waiting 60s (attempt 1)...
  ✓ MSNBC        [climate change      ] +0 new stories
  ✓ AP           [cli

In [6]:
# Balanced sampling and full text scraping
# Full metadata stays in mediacloud_articles.csv for later use

import trafilatura

df = pd.read_csv('mediacloud_articles.csv')
print(f"Total articles available: {len(df)}")

# Balanced sample: 100 articles per source per topic
ARTICLES_PER_GROUP = 100

sampled = df.groupby(['topic', 'source_name']).apply(
    lambda x: x.sample(min(len(x), ARTICLES_PER_GROUP), random_state=42)
).reset_index(drop=True)

print(f"Sampled for scraping: {len(sampled)} articles")
print(f"\nBy source:")
print(sampled.groupby('source_name').size())
print(f"\nBy topic:")
print(sampled.groupby('topic').size())

# Save sampled URLs (before scraping) so we know which ones we've done
sampled.to_csv('mediacloud_sampled.csv', index=False)
print(f"\nSaved sample to mediacloud_sampled.csv")
print(f"(Full {len(df)} articles still in mediacloud_articles.csv for later)")

est_minutes = len(sampled) * 1.5 / 60
print(f"\nEstimated scraping time: ~{est_minutes:.0f} minutes")

Total articles available: 19554
Sampled for scraping: 2424 articles

By source:
source_name
AP          500
BBC         500
CNN         500
Fox News    479
Newsmax     445
dtype: int64

By topic:
topic
AI             500
Climate        424
Economy        500
Immigration    500
Iran           500
dtype: int64

Saved sample to mediacloud_sampled.csv
(Full 19554 articles still in mediacloud_articles.csv for later)

Estimated scraping time: ~61 minutes


C:\Users\woodj\AppData\Local\Temp\ipykernel_39356\1097605797.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = df.groupby(['topic', 'source_name']).apply(


In [7]:
# Scrape full text for sampled articles

sampled = pd.read_csv('mediacloud_sampled.csv')
print(f"Scraping {len(sampled)} articles...\n")

full_texts = []
titles_extracted = []

for i, row in sampled.iterrows():
    if i % 50 == 0:
        print(f"Progress: {i}/{len(sampled)} ({i/len(sampled)*100:.0f}%)")
    
    try:
        html = trafilatura.fetch_url(row['url'])
        if html:
            text = trafilatura.extract(html)
            meta = trafilatura.extract_metadata(html)
            full_texts.append(text)
            titles_extracted.append(meta.title if meta else None)
        else:
            full_texts.append(None)
            titles_extracted.append(None)
    except Exception as e:
        full_texts.append(None)
        titles_extracted.append(None)
    
    time.sleep(0.5)

sampled['full_text'] = full_texts
sampled['title_scraped'] = titles_extracted
sampled['text_length'] = sampled['full_text'].apply(lambda x: len(x) if x else 0)

# Filter to successful scrapes
success = sampled[sampled['text_length'] > 100].copy()
print(f"\n\nSuccessfully scraped: {len(success)}/{len(sampled)} ({len(success)/len(sampled)*100:.0f}%)")

print(f"\nSuccess by source:")
print(sampled.groupby('source_name').apply(lambda x: (x['text_length'] > 100).sum()))

# Save
success.to_csv('mediacloud_articles_full.csv', index=False)
print(f"\nSaved to mediacloud_articles_full.csv")

Scraping 2424 articles...

Progress: 0/2424 (0%)
Progress: 50/2424 (2%)
Progress: 100/2424 (4%)
Progress: 150/2424 (6%)
Progress: 200/2424 (8%)
Progress: 250/2424 (10%)
Progress: 300/2424 (12%)
Progress: 350/2424 (14%)
Progress: 400/2424 (17%)
Progress: 450/2424 (19%)
Progress: 500/2424 (21%)
Progress: 550/2424 (23%)
Progress: 600/2424 (25%)
Progress: 650/2424 (27%)
Progress: 700/2424 (29%)
Progress: 750/2424 (31%)
Progress: 800/2424 (33%)
Progress: 850/2424 (35%)
Progress: 900/2424 (37%)
Progress: 950/2424 (39%)
Progress: 1000/2424 (41%)
Progress: 1050/2424 (43%)
Progress: 1100/2424 (45%)
Progress: 1150/2424 (47%)
Progress: 1200/2424 (50%)
Progress: 1250/2424 (52%)
Progress: 1300/2424 (54%)
Progress: 1350/2424 (56%)
Progress: 1400/2424 (58%)
Progress: 1450/2424 (60%)
Progress: 1500/2424 (62%)
Progress: 1550/2424 (64%)
Progress: 1600/2424 (66%)
Progress: 1650/2424 (68%)
Progress: 1700/2424 (70%)
Progress: 1750/2424 (72%)
Progress: 1800/2424 (74%)
Progress: 1850/2424 (76%)
Progress: 190

C:\Users\woodj\AppData\Local\Temp\ipykernel_39356\972789574.py:38: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(sampled.groupby('source_name').apply(lambda x: (x['text_length'] > 100).sum()))



Saved to mediacloud_articles_full.csv


In [14]:
# Diagnose and clean data

df = pd.read_csv('mediacloud_articles_full.csv')

print("=== Data Overview ===")
print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

print("\n=== Check for issues ===")
print(f"Null full_text: {df['full_text'].isna().sum()}")
print(f"Empty full_text: {(df['text_length'] == 0).sum()}")
print(f"Short articles (<100 chars): {(df['text_length'] < 100).sum()}")

print("\n=== Language distribution ===")
print(df['language'].value_counts())

print("\n=== By source ===")
print(df.groupby('source_name').size())

print("\n=== Sample of non-English articles ===")
non_english = df[df['language'] != 'en']
if len(non_english) > 0:
    print(non_english[['source_name', 'language', 'title']].head(10))

=== Data Overview ===
Total rows: 2423
Columns: ['topic', 'keyword', 'source_name', 'source_id', 'title', 'url', 'publish_date', 'language', 'full_text', 'title_scraped', 'text_length']

=== Check for issues ===
Null full_text: 0
Empty full_text: 0
Short articles (<100 chars): 0

=== Language distribution ===
language
en    2334
es      18
sw      15
vi       8
tr       8
fr       6
tl       5
pl       5
id       3
hr       3
pt       3
zh       3
ar       2
ms       2
uk       2
ru       2
ky       1
ja       1
bn       1
az       1
Name: count, dtype: int64

=== By source ===
source_name
AP          500
BBC         500
CNN         499
Fox News    479
Newsmax     445
dtype: int64

=== Sample of non-English articles ===
   source_name language                                              title
1           AP       es  La UE amenaza con obligar a Meta a restablecer...
5           AP       es  Modi presenta a India como centro mundial de i...
23          AP       es  Adiós al CIA World F

In [15]:
# Clean data: filter to English, fix formatting and encoding issues

import csv

# Read with explicit encoding
df = pd.read_csv('mediacloud_articles_full.csv', encoding='utf-8')
print(f"Before cleaning: {len(df)} articles")

# Filter to English only
df_clean = df[df['language'] == 'en'].copy()
print(f"After English filter: {len(df_clean)} articles")

# Remove articles with no/short text
df_clean = df_clean[df_clean['text_length'] >= 200].copy()
print(f"After min length filter (200 chars): {len(df_clean)} articles")

# Remove duplicates by URL
df_clean = df_clean.drop_duplicates(subset='url').copy()
print(f"After deduplication: {len(df_clean)} articles")

# Fix text fields
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Fix common encoding issues
    text = text.replace('â€™', "'").replace('â€œ', '"').replace('â€', '"')
    text = text.replace('â€"', '-').replace('â€"', '-').replace('â€¦', '...')
    text = text.replace('Â', ' ')
    # Remove any remaining non-ASCII
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Remove newlines and normalize whitespace
    text = ' '.join(text.split())
    return text

df_clean['full_text'] = df_clean['full_text'].apply(clean_text)
df_clean['title'] = df_clean['title'].apply(clean_text)
df_clean['title_scraped'] = df_clean['title_scraped'].apply(clean_text)
print("Fixed text formatting and encoding")

print("\n=== Final distribution ===")
print("\nBy source:")
print(df_clean.groupby('source_name').size())
print("\nBy topic:")
print(df_clean.groupby('topic').size())

# Save as CSV with QUOTE_ALL - this properly handles commas and quotes in text
df_clean.to_csv('mediacloud_articles_clean.csv', index=False, quoting=csv.QUOTE_ALL)
print(f"\nSaved to mediacloud_articles_clean.csv")

# Verify CSV loads correctly
test = pd.read_csv('mediacloud_articles_clean.csv')
print(f"Verified CSV: {len(test)} rows, {len(test.columns)} columns")
print(f"Columns: {list(test.columns)}")

Before cleaning: 2423 articles
After English filter: 2334 articles
After min length filter (200 chars): 2334 articles
After deduplication: 2334 articles
Fixed text formatting and encoding

=== Final distribution ===

By source:
source_name
AP          484
BBC         429
CNN         497
Fox News    479
Newsmax     445
dtype: int64

By topic:
topic
AI             458
Climate        421
Economy        497
Immigration    499
Iran           459
dtype: int64

Saved to mediacloud_articles_clean.csv
Verified CSV: 2334 rows, 11 columns
Columns: ['topic', 'keyword', 'source_name', 'source_id', 'title', 'url', 'publish_date', 'language', 'full_text', 'title_scraped', 'text_length']
